## 1. Setup
Imports, the Groq client and the embedding model `all-MiniLM-L6-v2`, a small sentence-transformer that turns text into 384-dimensional vectors. Groq exposes an OpenAI-compatible API, so the client is the `openai` SDK pointed at Groq's base URL. It reads `GROQ_API_KEY` from `.env`.

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()
client = OpenAI(base_url="https://api.groq.com/openai/v1",
                     api_key=os.getenv("GROQ_API_KEY"))

model = SentenceTransformer("all-MiniLM-L6-v2")

## 2. Build the index
### Chunking function
`chunk_text` splits text into 150-word chunks that overlap by 30 words.

In [ ]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    result = []
    for start in range(0, len(words), chunk_size - overlap):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        result.append({"chunk_id": len(result), "text": chunk})
        if end >= len(words):
            break
    return result

### Load the document
Reads `doc.txt` (the gym and health handbook) into `text`.

In [ ]:
with open("doc.txt", encoding= "utf-8") as f:
    text = f.read()

### Chunk the document
Splits the text into overlapping chunks and prints how many there are.

In [ ]:
chunks = chunk_text(text)
print(f"{len(chunks)} Chunks")

### Embed the chunks
Encodes every chunk once and stores the vectors in `chunk_vector`, which `retrieve` searches.

In [ ]:
chunk_vector = model.encode([c["text"] for c in chunks])

## 3. Retrieval
- `cosine_similarity`: how similar two vectors are (1 = same direction).
- `retrieve`: embeds the query and returns the `k` most similar chunks.
- `expand_with_neighbors`: adds the chunks on either side of each result.

In [ ]:
def cosine_similarity(a, b):
    return (np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve(query, k=3):
    query_vec = model.encode(query)
    scores = []
    for ch_v, ch in zip(chunk_vector, chunks):
        sim = float(cosine_similarity(ch_v, query_vec))
        scores.append({"id": ch["chunk_id"], "score": sim, "chunk": ch})
        
    s_scores = sorted(scores, key=lambda x: x["score"], reverse=True)
    return s_scores[:k]

def expand_with_neighbors(results, window=1):
    ids = set()
    for r in results:
        center = r["id"]
        for neighbor in range(center - window, center + window + 1):
            if 0 <= neighbor < len(chunks):
                ids.add(neighbor)
    return [chunks[i] for i in sorted(ids)]

## 4. Generation
### RAG answer function
`answer(query)` retrieves the top 3 chunks, labels each one `[chunk N]` in the prompt, and asks `openai/gpt-oss-120b` on Groq to answer using only that context. The prompt requires:
- a citation after every sentence, in the format `[chunk N]`;
- the exact reply "I couldn't find that in the documents." when the context doesn't contain the answer.

The reply is normalized with `unicodedata` NFKC, because the model inserts narrow no-break spaces that break text matching. Citations are then parsed with `CITE_RE`. `answer` returns a dict:

| Key | What it holds |
|---|---|
| `answer` | The model's reply |
| `retrieved_ids` | Chunk IDs that went into the prompt |
| `cited_ids` | Chunk IDs the reply cites |
| `invalid_citations` | Cited but never retrieved (hallucinated IDs) |
| `uncited_retrieved` | Retrieved but not cited |
| `no_answer` | `True` if the model refused |
| `sources` | Text of each cited chunk, by ID |

In [ ]:
import re
import unicodedata

system_instructions = """Answer the question using ONLY the context below.
If the context does not contain the answer, reply exactly: "I couldn't find that in the documents."
Do not use outside knowledge, even if you think the context is wrong.
After each sentence, cite the chunk(s) that support it as [chunk N], one bracket per chunk, e.g. [chunk 3][chunk 7].
Do not mention "the context" or "the documents" in your answer; just answer."""

CITE_RE = re.compile(r"\[chunk (\d+)\]")

def answer(query):
    results = retrieve(query, k=3)
    context = "\n\n".join(f"[chunk {r['id']}]\n{r['chunk']['text']}" for r in results)
    prompt = f"{system_instructions}\n\n<context>\n{context}\n</context>\n\nQuestion: {query}"
    
    reply = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": prompt}],
    )
    text = unicodedata.normalize("NFKC", reply.choices[0].message.content)
    
    retrieved = {r["id"] for r in results}
    cited = {int(m) for m in CITE_RE.findall(text)}
    return{
        "answer": text,
        "retrieved_ids": sorted(retrieved),
        "cited_ids": sorted(cited),
        "invalid_citations": sorted(cited - retrieved),   # hallucinated IDs
        "uncited_retrieved": sorted(retrieved - cited),   # useful retrieval signal
        "no_answer": text.strip().startswith("I couldn't find that"),
        "sources": {r["id"]: r["chunk"]["text"] for r in results if r["id"] in cited},
    }

### Try a question
Runs one question through the full RAG pipeline. The cell is switched off with `%%script false` so that "run all" doesn't spend an API call; delete that line to run it.

In [ ]:
%%script false --no-raise-error
answer("how much protein should I eat per day?")

## 5. Evaluation
### Keyword lookup
`find_chunk` returns the IDs of chunks that contain a keyword. It was used to find the gold chunks below, and to check that the refusal questions really have no answer in the doc. Example: the chunks that mention "pregnant".

In [ ]:
def find_chunk(keyword):
    return [c["chunk_id"] for c in chunks if keyword.lower() in c["text"].lower()]

find_chunk("pregnant")

### Evaluation set
Two sets of test questions, each labelled with what the model should do (`expected`):
- `eval_set`: ten questions the document answers, with `expected: "answer"`. Each has its gold chunk IDs (the chunks that contain the answer) and `key_words`: phrases that should appear in a correct answer, used by `score` in section 7 to give a first label.
- `refusal_set`: nine questions the document doesn't answer, so the model should reply "I couldn't find that in the documents." They have no gold chunks and `expected: "refuse"`. Three are out of scope (nothing related in the doc). Six are near misses: the topic is in the doc but the specific answer isn't, so retrieval returns relevant-looking chunks and the model is tempted to stretch.

`full_set` combines both, for evaluating answers and refusals together.

A planned refusal question, "how much does the Titan plan cost per year?", was replaced. The model can answer it correctly by multiplying the monthly price (60 × 12 = 720 JOD), so it doesn't test refusal.

In [ ]:
eval_set = [
    {"query": "how much protein should I eat per day?",                 "gold": [27, 28]},
    {"query": "can I bring a friend with me to the gym?",               "gold": [48]},

    {"query": "why are my legs still sore two days after squats?",      "gold": [36, 37]},
    {"query": "my weight isn't changing, how do I know if I'm losing fat?", "gold": [44]},
    {"query": "how many days off should a beginner take each week?",    "gold": [36]},

    {"query": "can I use my student discount on the Titan plan?",       "gold": [47]},
    {"query": "is it fine to do HIIT every day?",                       "gold": [24]},
    {"query": "should I hold long stretches before lifting heavy?",     "gold": [21]},

    {"query": "what does error IH-409 mean?",                           "gold": [51]},

    {"query": "is creatine bad for my kidneys?",                        "gold": [32]},
]

key_words = {
    "how much protein should I eat per day?":                   ["1.6"],
    "can I bring a friend with me to the gym?":                 ["Silver", "Titan"],
    "why are my legs still sore two days after squats?":        ["72", "DOMS"],
    "my weight isn't changing, how do I know if I'm losing fat?": ["waist"],
    "how many days off should a beginner take each week?":      ["one or two", "1 or 2", "1-2", "1 to 2"],
    "can I use my student discount on the Titan plan?":         ["not apply", "doesn't apply", "does not apply", "not available", "cannot"],
    "is it fine to do HIIT every day?":                         ["not recommended", "two sessions", "2 sessions"],
    "should I hold long stretches before lifting heavy?":       ["not recommended", "reduce"],
    "what does error IH-409 mean?":                             ["full"],
    "is creatine bad for my kidneys?":                          ["safe", "does not damage", "doesn't damage"],
}

for q in eval_set:
    q["key_words"] = key_words[q["query"]]

for q in eval_set:
    q["expected"] = "answer"
    
refusal_set = [
    # Out of scope: nothing related in the doc. These should be easy.
    {"query": "what's the best creatine brand to buy?",          "gold": [], "expected": "refuse"},
    {"query": "who won the last football World Cup?",            "gold": [], "expected": "refuse"},
    {"query": "how do I reset my iPhone?",                       "gold": [], "expected": "refuse"},

    # Near misses: the topic is in the doc, the specific answer isn't.
    # Retrieval will return relevant-looking chunks, so the model is tempted to stretch.
    {"query": "what does error IH-777 mean?",                    "gold": [], "expected": "refuse"},  # made-up code, close to IH-409
    {"query": "how much protein should a 12-year-old eat?",      "gold": [], "expected": "refuse"},  # doc covers adults who lift
    {"query": "is creatine safe to take while pregnant?",        "gold": [], "expected": "refuse"},  # doc covers creatine + kidneys only
    {"query": "can I bring my dog to the gym?",                  "gold": [], "expected": "refuse"},  # doc covers human guests
    {"query": "is there a discount if I pay for a whole year upfront?",     "gold": [], "expected": "refuse"}, 
    {"query": "is the gym open on public holidays?","gold": [], "expected": "refuse"},
]

full_set = eval_set + refusal_set

### Recall@k
For each question, checks whether any gold chunk appears in the top `k` results. Prints the misses and the overall recall score. Run it on `eval_set`, not `full_set`: the refusal questions have no gold chunks, so they would always count as misses.

In [ ]:
def recall_at_k(eval_set, k=3):    
    count = 0
    for q in eval_set:
        retrieved_ids = [r["id"] for r in retrieve(q["query"], k=k)]
        if set(q["gold"]) & set(retrieved_ids):
            count += 1
        else:
            print(f"MISS: {q['query']}  gold={q['gold']}  got={retrieved_ids}")

    print(f"recall@{k} = {count / len(eval_set):.2f}")

recall_at_k(eval_set, k=3)

### v0 answer evaluation (Gemini, k=3)
Results from the v0 baseline, before citations and the refusal set. The v1 results are in section 7.

1. How much protein should I eat per day? — **correct**
2. Can I bring a friend with me to the gym? — **false refusal**
3. Why are my legs still sore two days after squats? — **correct**
4. My weight isn't changing, how do I know if I'm losing fat? — **correct**
5. How many days off should a beginner take each week? — **false refusal**
6. Can I use my student discount on the Titan plan? — **correct**
7. Is it fine to do HIIT every day? — **correct**
8. Should I hold long stretches before lifting heavy? — **correct**
9. What does error IH-409 mean? — **correct**
10. Is creatine bad for my kidneys? — **correct**

**Result:** 8 correct, 2 false refusals, 0 wrong

## 6. Interactive search
`interactive_search()` asks for a question and prints the top 3 chunks with their similarity scores, until you enter `q`. Defining it doesn't start the loop; call `interactive_search()` to run it.

In [ ]:
def interactive_search():
    while True:
        query = input("\nQuestion (or 'q' to quit): ")
        if query == 'q':
            break
        for r in retrieve(query, k=3):
            print(f"\n[{r['id']}] {r['score']:.3f}\n{r['chunk']['text'][:300]}...")

## 7. v1 answer evaluation
### Run and score
Runs every question in `full_set` through `answer()` and labels the result with `score`:
- **wrong** if the answer cites a chunk that was never retrieved;
- for refusal questions: **correct refusal** if the model refused, otherwise **wrong**;
- for answer questions: **false refusal** if the model refused, **correct** if any of the question's `key_words` appears in the answer, otherwise **wrong**.

The key-word check only gives a first label; every answer was then read and checked by hand. The loop waits 5 seconds between calls to stay under Groq's free-tier limit.

In [ ]:

import time
from collections import Counter

def score(q, result):
    refused = "couldn't find" in result["answer"].lower()
    if result["invalid_citations"]:          # cited a chunk it never got
        return "wrong"
    if q["expected"] == "refuse":
        return "correct refusal" if refused else "wrong"
    if refused:
        return "false refusal"
    text = result["answer"].lower()
    if any(w.lower() in text for w in q["key_words"]):
        return "correct"
    return "wrong"

results = []
for q in full_set:
    r = answer(q["query"])
    label = score(q, r)
    results.append({"query": q["query"], "label": label, "answer": r["answer"], "cited": r["cited_ids"]})
    print(f"{label:16} | {q['query']}")
    time.sleep(5)    # stay under the free API limit

print(Counter(r["label"] for r in results))

### Hand corrections
Applies the fixes from reading the answers by hand:
- drops "how much does the Titan plan cost per year?", which was in the run before it was replaced (see the evaluation set);
- relabels the student-discount question as **correct**, after checking its answer by hand.

In [ ]:
results = [r for r in results if r["query"] != "how much does the Titan plan cost per year?"]

for r in results:
    if r["query"] == "can I use my student discount on the Titan plan?":
        r["label"] = "correct"

print(len(results))
print(Counter(r["label"] for r in results))

### Results (gpt-oss-120b on Groq, k=3)

|                   | v0 (Gemini) | v1 (gpt-oss-120b) |
|-------------------|-------------|------------------|
| correct           | 8 / 10      | 8 / 10           |
| false refusal     | 2           | 1                |
| wrong             | 0           | 1                |
| correct refusals  | n/a         | 9 / 9            |
| invalid citations | n/a         | 0                |

Findings:
- **"How many days off should a beginner take each week?" was answered wrongly.** The chunk with the answer (chunk 36: beginners should take at least one or two full rest days per week) ranked 5th, so with k=3 the model never saw it. Instead of refusing, the model worked out "four days off" from the 3-day training program chunks. A retrieval miss turned into a made-up answer, not a refusal. Chunk 36 mixes sleep tips with rest days, which likely weakens its match. To re-test in v2 and v3.
- **1 false refusal:** "can I bring a friend with me to the gym?" (the doc uses the word "guest").
- **Recall@3 is unchanged from v0**, because v1 did not change retrieval.
- **Caveat:** v1 changed two things at once, the prompt and the model, so the change from v0 can't be credited to either one alone.